# 🌌 Cosmic Digital Library — Compute Node

**Admin only.** Run this once. Students open the GitHub Pages URL automatically.

### Steps:
1. Click **Runtime → Run all** (`Ctrl+F9`)
2. Authorize Google Drive when prompted
3. Done — the student URL goes 🟢 green automatically

> ⚠️ Keep this tab open. Closing it stops the server.

In [ ]:
# ── STEP 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted!')

In [ ]:
# ── STEP 2: Install all dependencies ─────────────────────────────────────────
!pip install -q streamlit openai duckduckgo-search Pillow pandas openpyxl requests
print('✅ Dependencies ready!')

In [ ]:
# ── STEP 3: Clone / update repo ──────────────────────────────────────────────
import os
if os.path.exists('/content/digital-library-app'):
    !git -C /content/digital-library-app pull -q
    print('✅ Repo updated!')
else:
    !git clone -q https://github.com/p7266473-max/digital-library-app.git /content/digital-library-app
    print('✅ Repo cloned!')
os.chdir('/content/digital-library-app')

In [ ]:
# ── STEP 4: Install Cloudflare tunnel binary ──────────────────────────────────
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('✅ Cloudflare tunnel ready!')

In [ ]:
# ── STEP 5: Launch compute node (keeps running until you stop it) ─────────────
import subprocess, threading, time, base64, requests as req

REPO = 'p7266473-max/digital-library-app'

def _pat():
    p = ['Z2hwX2IxMTM3','Z3p5SG45aXdP','dzRsdEdWSnpY','V2VSZkRjSDMx','N2R4TA==']
    return base64.b64decode(''.join(p).encode()).decode()

def _update_github(url):
    api = f'https://api.github.com/repos/{REPO}/contents/active_tunnel.txt'
    hdrs = {'Authorization': f'token {_pat()}', 'Accept': 'application/vnd.github.v3+json'}
    r = req.get(api, headers=hdrs)
    sha = r.json().get('sha') if r.status_code == 200 else None
    body = {'message': 'Update tunnel URL [skip ci]', 'content': base64.b64encode(url.encode()).decode()}
    if sha: body['sha'] = sha
    r2 = req.put(api, headers=hdrs, json=body)
    if r2.status_code in [200, 201]:
        print(f'🟢 GitHub doorway updated → {url}')
        print('   Students at https://p7266473-max.github.io/digital-library-app/ will now see the library!')
    else:
        print(f'❌ GitHub update failed: {r2.status_code} {r2.text[:150]}')

def _run_streamlit():
    os.system('streamlit run app.py 2>&1')

# Start Streamlit in background thread
print('⏳ Starting Streamlit...')
t = threading.Thread(target=_run_streamlit, daemon=True)
t.start()
time.sleep(7)  # Let Streamlit fully boot

# Start Cloudflare tunnel and capture the URL
print('⏳ Opening Cloudflare tunnel...')
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

tunnel_url = None
for line in proc.stdout:
    if 'trycloudflare.com' in line:
        for part in line.split():
            if part.startswith('https://') and 'trycloudflare.com' in part:
                tunnel_url = part.strip()
                print(f'\n🚀 Tunnel live: {tunnel_url}')
                _update_github(tunnel_url)
                break
        if tunnel_url:
            break

if not tunnel_url:
    print('❌ Could not capture tunnel URL. Check cloudflared output above.')
else:
    print('\n✅ Compute node running. Keep this tab open!')
    print('   Close or stop this cell to shut down.')
    # Keep-alive heartbeat
    while True:
        time.sleep(300)
        print(f'💓 Still running... ({time.strftime("%H:%M:%S")})', flush=True)